In [2]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 221.1 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.1.0-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.1.0-py3-none-any.whl (319 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp313-cp313-linux_x86_64.whl size=4711560 sha256=85ea857935a381c0b5497c9906811d6901bd32b83ffe1657659d87dac91e0369
  Stored in directory: /root/.cache/pip/wheels/59/06/3f/c95dbba0df6e58ba6ed18220c1aea1e96042802139df3674ef
Successfully built fasttext


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.models import Sequential
from sklearn.metrics import classification_report
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout, SpatialDropout1D


In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from collections import Counter
from tensorflow.keras.preprocessing.text import Tokenizer
import fasttext
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.utils.class_weight import compute_class_weight


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
pd.set_option('display.max_colwidth', None)

file_path= "/content/drive/MyDrive/Colab Notebooks/digikala_preprocessed.csv"
# file_path= "../data/digikala_preprocessed.csv"

df = pd.read_csv(file_path, encoding="utf-8")

df.head()

,comments,label
0,ازش راضی داخل شگفت‌انگیز خرید عالی هرک بوش فهمید خوشش,2
1,بافت اسفنج کاملاً نرم خوب خوب بلند خیسشم بهتر,2
2,شامپو ضد شوره شامپو راضی ضعیف ترشو خرید,2
3,بسیار شیک خاص طرفدار همیشگی به‌شخصه دریا باشگاه هرجا پوشید هرنظری عالی,2
4,محافظ خوبی هست,2


In [8]:
print(len(df))

df.dropna(inplace=True)

print(len(df))


1962253
1961295


In [9]:
from sklearn.model_selection import train_test_split

X = df["comments"]
y = df["label"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Training data: {len(X_train)} samples")
print(f"Validation data: {len(X_val)} samples")
print(f"Testing data: {len(X_test)} samples")

Training data: 1569036 samples
Validation data: 196129 samples
Testing data: 196130 samples


In [10]:
test_data = pd.DataFrame({"comments": X_test, "label": y_test})

test_data.to_csv("/content/drive/MyDrive/Colab Notebooks/test_data.csv", index=False, encoding="utf-8")
# test_data.to_csv("../data/test_data.csv", index=False, encoding="utf-8")

print("Test data saved.")
print("Test data shape:", test_data.shape)

Test data saved.
Test data shape: (196130, 2)


In [11]:
all_words = [word for comment in X_train for word in comment.split()]
word_counts = Counter(all_words)

total_word_occurrences = len(all_words)
unique_words = len(word_counts)

print(f"Total word occurrences: {total_word_occurrences}")
print(f"Total unique words: {unique_words}")

sorted_word_counts = word_counts.most_common()

for vocab_size in [1000, 5000, 10000, 15000, 20000, 25000, 30000]:
    covered_occurrences = sum(count for _, count in sorted_word_counts[:vocab_size])
    coverage_percentage = (covered_occurrences / total_word_occurrences) * 100
    print(f"vocab_size={vocab_size}: covers {coverage_percentage:.2f}% of all word occurrences")

Total word occurrences: 16000786
Total unique words: 138320
vocab_size=1000: covers 81.30% of all word occurrences
vocab_size=5000: covers 94.77% of all word occurrences
vocab_size=10000: covers 97.26% of all word occurrences
vocab_size=15000: covers 98.13% of all word occurrences
vocab_size=20000: covers 98.57% of all word occurrences
vocab_size=25000: covers 98.84% of all word occurrences
vocab_size=30000: covers 99.01% of all word occurrences


In [12]:
max_words = 20000

tkn = Tokenizer(num_words=max_words, oov_token="<OOV>")
tkn.fit_on_texts(X_train)

In [13]:
file_path = "/content/drive/MyDrive/Colab Notebooks/cc.fa.300.bin"
# file_path = "../models/cc.fa.300.bin"
model = fasttext.load_model(file_path)

In [17]:
word_index = tkn.word_index
embedding_dim = 300

embedding_matrix = np.zeros((max_words, embedding_dim))

for word, i in word_index.items():
    if i < max_words:
        embedding_vector = model.get_word_vector(word)
        embedding_matrix[i] = embedding_vector

In [20]:
sentence_lengths = [len(seq) for seq in tkn.texts_to_sequences(X_train)]

sentence_lengths = np.array(sentence_lengths)

for length in [44, 45, 46, 47, 48, 49, 50, 51, 52]:
    percent = np.mean(sentence_lengths <= length) * 100
    print(f"Percentage of sentences with length <= {length}: {percent:.2f}%")

Percentage of sentences with length <= 44: 98.80%
Percentage of sentences with length <= 45: 98.89%
Percentage of sentences with length <= 46: 98.96%
Percentage of sentences with length <= 47: 99.02%
Percentage of sentences with length <= 48: 99.09%
Percentage of sentences with length <= 49: 99.14%
Percentage of sentences with length <= 50: 99.19%
Percentage of sentences with length <= 51: 99.25%
Percentage of sentences with length <= 52: 99.29%


In [23]:
seq = tkn.texts_to_sequences(X_train)
max_len = 50

padded_docs = pad_sequences(seq, padding='post',truncating='post', maxlen=max_len)

print(padded_docs)

[[  6 220 628 ...   0   0   0]
 [ 99   6  65 ...   0   0   0]
 [ 71 299  23 ...   0   0   0]
 ...
 [299   7  38 ...   0   0   0]
 [  2   2   5 ...   0   0   0]
 [ 96  94   3 ...   0   0   0]]


In [25]:
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(1.4016696369946533), np.int64(1): np.float64(1.319607709561764), np.int64(2): np.float64(0.6541231324149972)}
